## 데이터 살펴보기

In [1]:
import pandas as pd

In [2]:
train = pd.read_csv("train.csv", encoding="cp949")
test= pd.read_csv("test.csv", encoding="cp949")

train.head()

,sentence,label
0,오늘은 누구랑 같이 밥을 먹어야 할까?,불안
1,난 남들 다 받는 과외를 못 받아서 공부를 못하는 것 같아.,슬픔
2,오늘 친구가 몸이 너무 안 좋아서 친구 잘못으로 내가 대신 혼났어.,상처
3,우리 주위에는 불치병으로 인하여 어려운 분들이 많은 것 같아.,상처
4,아이들이 사춘기가 되니 예전처럼 나에게 먼저 말을 걸지 않고 자꾸 나를 피하려고 해.,당황


In [3]:
label_tags = train['label'].unique()
label_tags

array(['불안', '슬픔', '상처', '당황', '분노', '기쁨'], dtype=object)

In [4]:
print(train.iloc[:7].to_markdown())

|    | sentence                                                                          | label   |
|---:|:----------------------------------------------------------------------------------|:--------|
|  0 | 오늘은 누구랑 같이 밥을 먹어야 할까?                                              | 불안    |
|  1 | 난 남들 다 받는 과외를 못 받아서 공부를 못하는 것 같아.                           | 슬픔    |
|  2 | 오늘 친구가 몸이 너무 안 좋아서 친구 잘못으로 내가 대신 혼났어.                   | 상처    |
|  3 | 우리 주위에는 불치병으로 인하여 어려운 분들이 많은 것 같아.                       | 상처    |
|  4 | 아이들이 사춘기가 되니 예전처럼 나에게 먼저 말을 걸지 않고 자꾸 나를 피하려고 해. | 당황    |
|  5 | 딸이 결혼을 안 하려고 해서 집사람이 화가 났어.                                    | 분노    |
|  6 | 학원 친구 중에 나보다 뛰어난 애가 눈에 밟혀서 화가 나.                            | 상처    |


In [5]:
print(test.iloc[:7].to_markdown())

|    | sentence                                                                                             | label   |
|---:|:-----------------------------------------------------------------------------------------------------|:--------|
|  0 | 수험생인 아들과 기분 전환 겸 집 앞 산을 등산하다가 아들이 발목을 접질려서 너무 마음 아파.            | 당황    |
|  1 | 내 장례식만큼은 내가 원하는 대로 해 달라고 말했는데 자식들은 들은 체도 안 해. 속상해.                | 상처    |
|  2 | 친구가 지난해에 이혼을 했어. 양육비를 월 오십만 원씩 주기로 하고 아이는 부인이 키우기로 했다는 거야. | 당황    |
|  3 | 학교에 늦어서 엄마 출근길에 나 좀 데려다 달라고 했는데 엄마가 화가 났을까?                           | 분노    |
|  4 | 가족들이 집으로 빨리 돌아오라고 했는데 늦게 가서 걱정만 시켰어.                                      | 슬픔    |
|  5 | 건강에 회의적이야.                                                                                   | 불안    |
|  6 | 은퇴하고 아픈 곳이 없지만 돈이 없어.                                                                 | 당황    |


## BERT 파생모델 TEST

In [6]:
import torch

input_ids = torch.LongTensor([[31, 51, 99], [15, 5, 0]])
attention_mask = torch.LongTensor([[1, 1, 1], [1, 1, 0]])
token_type_ids = torch.LongTensor([[0, 0, 1], [0, 1, 0]])

In [7]:
from transformers import BertTokenizerFast, BertModel
tokenizer_bert = BertTokenizerFast.from_pretrained("kykim/bert-kor-base")
model_bert = BertModel.from_pretrained("kykim/bert-kor-base")

Some weights of the model checkpoint at kykim/bert-kor-base were not used when initializing BertModel: ['cls.predictions.decoder.weight', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.decoder.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [8]:
output = model_bert(input_ids, attention_mask, token_type_ids)
print(output.keys())

seq_output = output.last_hidden_state
pooler_output = output.pooler_output

print(seq_output.shape)
print(pooler_output.shape)

odict_keys(['last_hidden_state', 'pooler_output'])
torch.Size([2, 3, 768])
torch.Size([2, 768])


In [9]:
text = ["목사님 주보랑 ppt(설교부분 제외) 보내드려요~", "문장이 주어지는 경우가 있다."]

token = tokenizer_bert.tokenize(text)
token_ids = tokenizer_bert.convert_tokens_to_ids(token)
print(token)
print(token_ids)

['목사', '##님', '주', '##보', '##랑', 'pp', '##t', '(', '설', '##교', '##부분', '제외', ')', '보내', '##드려요', '~', '문장', '##이', '주어', '##지는', '경우가', '있다', '.']
[35714, 8403, 6165, 8032, 8202, 25151, 8086, 2010, 4964, 8033, 14194, 15729, 2011, 14396, 16980, 2070, 31622, 8018, 20808, 14029, 15218, 13984, 2016]


In [10]:
token_ids = tokenizer_bert.encode(text, max_length=128, padding='max_length')

print(token_ids)

[2, 35714, 8403, 6165, 8032, 8202, 25151, 8086, 2010, 4964, 8033, 14194, 15729, 2011, 14396, 16980, 2070, 3, 31622, 8018, 20808, 14029, 15218, 13984, 2016, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [11]:
"""
https://misconstructed.tistory.com/80

- text : encoding 될 문장 (sequence, batch). str, List[str], List[List[str]] 을 입력으로 받을 수 있고, 이미 tokenize 된 경우(list of tokens) is_split_into_words=True 로 지정해줘야 한다.

- text_pair : text와 동일. 한 개의 문장이 아니라 sentence pair를 입력으로 주는 경우인 듯.

- add_special_tokens (True) : encoding을 할 때 special token을 추가할지 여부 

- padding (False) : True로 지정한 경우 batch 내에서 가장 긴 길이에 맞춰서 padding을 한다. False인 경우 padding을 하지 않고 encoding을 한다. 'max_length'로 지정하면 모델이 입력으로 받을 수 있는 최대의 길이에 맞춰서 padding을 한다. 

- truncation (False): True 인 경우 모델이 입력으로 받을 수 있는 최대의 길이에 맞춰서 Truncation을 한다. "only_first"인 경우 sentence pair로 입력이 주어지는 경우, 첫 번째 문장만 Truncate 한다. 반대로, "only_second"인 경우 두 번째 문장만 truncate 한다. False로 지정한 경우 trauncate을 하지 않는다.

- max_length : 모델이 입력으로 받을 최대의 입력 길이. 별도로 지정하지 않은 경우 모델의 config에 맞게 지정된다.

- is_split_into_words (False) : 입력이 이미 tokenize가 되었는지 여부. True로 지정한 경우, 입력이 이미 tokenize가 되었다고 가정하고 별도의 Tokenize를 수행하지 않는다. 

- return_tensors : "tf" 면 tf.constant, "pt" 면 torch.Tensor, "np" 면 np.ndarray 로 결과를 리턴한다. 

- return_token_type_ids : token type id를 제공할지 여부. 기본적으로 제공함.

- return_attention_mask : attention mask를 제공할지 여부. 기본적으로 제공함

- return_special_tokens_mask (False) : special token mask를 리턴할지 여부

- return_length (False) : encode된 결과의 길이를 리턴할지 여부

- verbose (True) : 추가적인 출력이나 warning 등을 출력할지 여부 리턴 → BachEncoding

- input_ids : token is 의 리스트

- token_type_ids : token type id의 리스트

- attention_mask : 모델에 의해서 attention이 수행되어야 하는 인덱스

- num_truncated_tokens : truncate된 token의 개수 

- special_token_mask : 0인 경우 일반 Token, 1인 경우 Special token

- length : return_length=True 인 경우, encoding 된 출력의 길이
"""

output = tokenizer_bert(text=text, max_length=128, padding='max_length',return_tensors='pt',
                        return_token_type_ids=True, return_attention_mask=True)

input_ids = output.input_ids
attention_mask = output.attention_mask
token_type_ids = output.token_type_ids

print(input_ids)
print(attention_mask)
print(token_type_ids)

tensor([[    2, 35714,  8403,  6165,  8032,  8202, 25151,  8086,  2010,  4964,
          8033, 14194, 15729,  2011, 14396, 16980,  2070,     3,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,  

In [12]:
print(input_ids.shape)
print(attention_mask.shape)
print(token_type_ids.shape)

torch.Size([2, 128])
torch.Size([2, 128])
torch.Size([2, 128])


In [13]:
output = model_bert(input_ids, attention_mask, token_type_ids)
print(output.keys())

seq_output = output.last_hidden_state
pooler_output = output.pooler_output

print(seq_output.shape)
print(pooler_output.shape)

odict_keys(['last_hidden_state', 'pooler_output'])
torch.Size([2, 128, 768])
torch.Size([2, 768])


In [14]:
input_ids = torch.LongTensor([[31, 51, 99], [15, 5, 0]])
attention_mask = torch.LongTensor([[1, 1, 1], [1, 1, 0]])
token_type_ids = torch.LongTensor([[0, 0, 1], [0, 1, 0]])

input_data = {'input_ids':input_ids, 'attention_mask':attention_mask, 'token_type_ids':token_type_ids}

output = model_bert(**input_data)
print(output.keys())


odict_keys(['last_hidden_state', 'pooler_output'])
